# 03 — Controlled within-Digits scaling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokhanturan/apsp-knn-benchmark/blob/main/notebooks/03_controlled_scaling.ipynb)

This notebook reruns the scale-and-sparsity experiment on the archived Digits-derived graphs. Five stratified samples are used at each size below 1,797; the full 1,797-sample dataset needs only one graph.

The reported crossover intervals are **environment-specific**, not universal vertex-count thresholds.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_NAME = "apsp-knn-benchmark"
REPO_URL = "https://github.com/gokhanturan/apsp-knn-benchmark.git"

# Colab: clone the repository if the notebook was opened directly from GitHub.
if Path('/content').exists() and not (Path('/content') / REPO_NAME).exists():
    subprocess.run(['git', 'clone', REPO_URL, str(Path('/content') / REPO_NAME)], check=True)

if (Path('/content') / REPO_NAME).exists():
    ROOT = Path('/content') / REPO_NAME
else:
    # Local/Jupyter execution from repo/notebooks or repo root.
    cwd = Path.cwd().resolve()
    ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

os.chdir(ROOT)
print('Repository root:', ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import sys, subprocess
from pathlib import Path
import pandas as pd
import numpy as np

GRAPH_DIR = ROOT / 'scaling_graphs'
OUT = ROOT / 'reproduced_results' / 'scaling'
OUT.mkdir(parents=True, exist_ok=True)
SIZES = [100,150,200,250,300,400,500,750,1000,1250,1500,1797]
K_VALUES = [5,10,20]
SEEDS = [0,1,2,3,4]


## Run the scaling timings

For publication reproduction leave `MAX_N = 1797`. Lower it for a quick smoke test.

In [ ]:
MAX_N = 1797
rows = []
for n in [v for v in SIZES if v <= MAX_N]:
    seed_ids = [0] if n == 1797 else SEEDS
    for seed_id in seed_ids:
        for k in K_VALUES:
            graph = GRAPH_DIR / f'digits_n{n}_k{k}_s{seed_id}.npz'
            out = OUT / f'scale_n{n}_k{k}_s{seed_id}.csv'
            cmd = [
                sys.executable, 'src/runtime_condition.py', '--graph', str(graph),
                '--dataset', 'digits_scaling', '--k', str(k), '--seed-id', str(seed_id),
                '--repeats', '6', '--warmups', '3', '--random-seed', '20260804',
                '--output', str(out)
            ]
            subprocess.run(cmd, check=True)
            part = pd.read_csv(out)
            part['n'] = n
            rows.append(part)

raw = pd.concat(rows, ignore_index=True)
raw.to_csv(OUT / 'scaling_runtime_raw.csv', index=False)
print('Rows:', len(raw))

## Summarize by seed and locate the Floyd-Warshall / repeated-Dijkstra ordering reversal

In [ ]:
seed_summary = (raw.groupby(['n','k','seed_id','algorithm'], as_index=False)
                  .agg(wall_median_s=('wall_time_s','median')))
condition = (seed_summary.groupby(['n','k','algorithm'], as_index=False)
             .agg(median_of_seed_medians=('wall_median_s','median')))

def reversal_for_k(k):
    p = condition[condition.k.eq(k)].pivot(index='n', columns='algorithm', values='median_of_seed_medians').sort_index()
    d_fast = p['dijkstra'] < p['floyd_warshall']
    for prev_n, n in zip(p.index[:-1], p.index[1:]):
        if (not bool(d_fast.loc[prev_n])) and bool(d_fast.loc[n]):
            return prev_n, n
    return None, None

rev = []
for k in K_VALUES:
    a,b = reversal_for_k(k)
    rev.append({'k':k, 'last_tested_n_with_floyd_not_slower':a, 'first_tested_n_with_dijkstra_faster':b})
reversal = pd.DataFrame(rev)
reversal

On the archived benchmark session the reported intervals were 150–200 vertices for `k=5` and `k=10`, and 200–250 vertices for `k=20`. A fresh Colab allocation can move these intersections.